# Forest Pattern Analysis

This notebook demonstrates the pyGuidos tools for forest pattern analysis on a binary Forest/Non-Forest map. We will apply four complementary tools:

- **MSPA** — Morphological Spatial Pattern Analysis: classifies forest pixels into structural categories (Core, Edge, Perforation, Islet, Branch, Loop, Bridge)
- **Fragmentation** — computes the proportion of forest pixels within a moving window, classifying landscape fragmentation
- **Accounting** — labels and classifies individual forest patches by size
- **RSS** — Raster Spatial Statistics: computes patch-based connectivity indices

**Input data**: Forest/Non-Forest map derived from **Corine Land Cover 2018** at **100m resolution**, Corsica, France.

**Analysis parameters**:
- MSPA: connectivity=8, edge_width=1, transition=1, int_ext=1
- Fragmentation: FAD method, window size 27x27 pixels (2.7km x 2.7km, ~729 ha)
- Accounting: 5 thresholds — 100, 1000, 10000, 100000, 1000000 pixels (10, 100, 1000, 10000, 100000 ha)
- RSS: no parameters required

## 1. Import Libraries and Define Paths

In [ ]:
import pyguidos as pg
from pyguidos import utils
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import rasterio

print(f"pyGuidos version: {pg.__version__}")

# --- Data paths ---
DATA_DIR = Path(pg.__file__).parent / "data"
fnf_tiff = DATA_DIR / "CLC2018_corsica_FNF.tif"

# --- Output directory: set your preferred output path here ---
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)
print(f"Output directory: {OUT_DIR.resolve()}")

## 2. Helper Function — Render GTB Colormap

pyGuidos output GeoTIFFs contain an embedded colour palette following the GuidosToolbox (GTB) convention. The function below reads the colormap directly from the output file and converts it to a matplotlib colormap, ensuring the visualisation matches the official GTB colour scheme exactly.

In [ ]:
def gtb_colormap(tiff_path):
    with rasterio.open(tiff_path) as src:
        data = src.read(1)
        cmap_dict = src.colormap(1)  # {value: (r, g, b, a)}
    
    # Build a 256-entry color array indexed directly by pixel value
    colors = np.zeros((256, 4), dtype=np.float32)
    for val, rgba in cmap_dict.items():
        if 0 <= val < 256:
            colors[val] = [c / 255.0 for c in rgba]
    
    cmap = ListedColormap(colors)
    
    # Use direct value mapping — each pixel value maps to its own color
    norm = plt.Normalize(vmin=0, vmax=255)
    
    return data, cmap, norm

## 3. MSPA — Morphological Spatial Pattern Analysis

MSPA classifies forest pixels into 7 structural categories based on their spatial context:

| Class | Description |
|---|---|
| **Core** | Interior forest pixels, away from the background |
| **Edge** | Forest pixels at the boundary with non-forest |
| **Perforation** | Forest pixels at the boundary with internal gaps |
| **Islet** | Small isolated forest patches too small to have core |
| **Branch** | Elongated forest connections ending at background |
| **Loop** | Elongated connections between edge pixels |
| **Bridge** | Elongated connections between core pixels |

**Parameters used**:
- `connectivity = 8` — diagonal connections allowed (recommended)
- `edge_width = 1` — edge zone of 1 pixel = 100m
- `transition = True` — enables transition zones between classes
- `int_ext = True` — distinguishes internal and external subclasses

In [ ]:
print("Running MSPA...")
mspa_result = pg.mspa(
    in_tiff=fnf_tiff,
    edge_width=1,
    connectivity=8,
    transition=True,
    int_ext=True,
    outdir=OUT_DIR,
    statists=True,
    stat_files=True,
    return_array=False,
    verb=False
)
print("\nMSPA completed.")

In [ ]:
# --- MSPA Statistics ---
print("Input pixel counts:")
for k, v in mspa_result.stats['input stats'].items():
    print(f"  {k:<20}: {v:>10}")

print("\nOutput pixel counts per class:")
tot_fg = mspa_result.stats['input stats']['foreground pxl']
class_freq = mspa_result.stats['output stats']['class freq']
for k, v in class_freq.items():
    pct = v / tot_fg * 100 if tot_fg > 0 else 0
    print(f"  {k:<25}: {v:>10} px  ({pct:6.2f}%)")

print(f"\n  Integral foreground : {mspa_result.stats['output stats']['integral foregr']:>10}")
print(f"  Porosity            : {mspa_result.stats['output stats']['porosity']:>10.4f}")

In [ ]:
# --- Visualise MSPA output with GTB colormap ---
mspa_tiff = mspa_result.stats['output paths']['path tif']
data, cmap, norm = gtb_colormap(mspa_tiff)

fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(data, cmap=cmap, norm=norm, interpolation='none')
ax.set_title('MSPA Result — Corsica\nCLC 2018 Forest/Non-Forest, 100m\nedge_width=1, connectivity=8', 
             fontsize=13, pad=15)
ax.axis('off')

# Legend — main MSPA classes with GTB colors
legend_patches = [
    mpatches.Patch(color=(0/255, 200/255, 0/255),   label='Core'),
    mpatches.Patch(color=(0/255, 0/255, 0/255),     label='Edge'),
    mpatches.Patch(color=(0/255, 0/255, 255/255),   label='Perforation'),
    mpatches.Patch(color=(160/255, 60/255, 0/255),  label='Islet'),
    mpatches.Patch(color=(255/255, 140/255, 0/255), label='Branch'),
    mpatches.Patch(color=(255/255, 255/255, 0/255), label='Loop'),
    mpatches.Patch(color=(255/255, 0/255, 0/255),   label='Bridge'),
    mpatches.Patch(color=(220/255, 220/255, 220/255), label='Background'),
    mpatches.Patch(facecolor='white', edgecolor='black', label='NoData'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

## 4. Fragmentation Analysis

Fragmentation analysis uses the **FAD** (Foreground Area Density) method with a **27x27 pixel moving window** (2.7km x 2.7km, approximately 729 ha at 100m resolution). Each foreground pixel receives a value [0-100] representing the proportion of forest pixels within the local neighbourhood, then classified into 5 fragmentation classes:

| Class | FAD range | Description |
|---|---|---|
| Rare | 0—10% | Very low forest density |
| Patchy | 10—40% | Low forest density |
| Transitional | 40—60% | Medium forest density |
| Dominant | 60—90% | High forest density |
| Interior | 90—100% | Very high forest density |

In [ ]:
print("Running Fragmentation (FAD, window=27)...")
frag_result = pg.frag(
    in_tiff=fnf_tiff,
    method='FAD',
    window_size=27,
    outdir=OUT_DIR,
    statists=True,
    stat_files=True,
    return_array=False,
    verb=False
)
print("\nFragmentation completed.")

In [ ]:
# --- Fragmentation Statistics ---
print("Input pixel counts:")
for k, v in frag_result.stats['input stats'].items():
    print(f"  {k:<20}: {v:>10}")

print("\nFragmentation class pixel counts:")
tot_fg = frag_result.stats['input stats']['foreground pxl']
class_freq = frag_result.stats['output stats']['class freq']
class_names = {
    '1 rare pxl'  : 'Rare',
    '2 patch pxl' : 'Patchy',
    '3 trans pxl' : 'Transitional',
    '4 domin pxl' : 'Dominant',
    '5 inter pxl' : 'Interior'
}
for key, name in class_names.items():
    v = class_freq[key]
    pct = v / tot_fg * 100 if tot_fg > 0 else 0
    print(f"  {name:<15}: {v:>10} px  ({pct:6.2f}%)")

print(f"\n  FAD average index : {frag_result.stats['output stats']['fad_av']:.2f}")
print(f"  AVcon index       : {frag_result.stats['output stats']['avcon']:.2f}")

In [ ]:
# --- Visualise Fragmentation output with GTB colormap + histogram ---
frag_tiff = frag_result.stats['output paths']['path tif']
data, cmap, norm = gtb_colormap(frag_tiff)
png_file = frag_result.stats['output paths']['path png']

from IPython.display import Image as IPImage
from PIL import Image as PILImage

fig, axes = plt.subplots(1, 2, figsize=(14, 10), 
                          gridspec_kw={'width_ratios': [1, 1.2]})

# Left — map
axes[0].imshow(data, cmap=cmap, norm=norm, interpolation='none')
axes[0].set_title('Fragmentation (FAD) — Corsica\nCLC 2018, 100m\nWindow: 27x27 pixels (2.7km x 2.7km)',
                  fontsize=12, pad=15)
axes[0].axis('off')

# Right — histogram
hist_img = PILImage.open(png_file)
axes[1].imshow(hist_img)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Accounting — Patch Size Classification

Accounting labels each individual forest patch and classifies it into one of 6 size classes based on the provided thresholds. At 100m resolution, 1 pixel = 1 ha, so the thresholds correspond directly to area in hectares:

| Class | Size range (pixels) | Color |
|---|---|---|
| 1 | 1 — 10 | Black |
| 2 | 11 — 100 | Red |
| 3 | 101 — 1,000 | Yellow |
| 4 | 1,001 — 10,000 | Orange |
| 5 | 10,001 — 100,000 | Brown |
| 6 | > 100,000 | Green |

In [ ]:
print("Running Accounting...")
acc_result = pg.acc(
    in_tiff=fnf_tiff,
    thresholds=[10, 100, 1000, 10000, 100000],
    outdir=OUT_DIR,
    statists=True,
    stat_files=True,
    return_array=False,
    verb=False
)
print("\nAccounting completed.")

In [ ]:
# --- Accounting Statistics ---
print("Input pixel counts:")
for k, v in acc_result.stats['input stats'].items():
    print(f"  {k:<20}: {v:>10}")

print("\nPatch counts per size class:")
patch_numb = acc_result.stats['output stats']['patch numb']
pxl_numb   = acc_result.stats['output stats']['pxl numb']
tot_fg     = acc_result.stats['input stats']['foreground pxl']

class_labels = [
    (103, 'Class 1 [1-10 ha]',             'Black'),
    (33,  'Class 2 [11-100 ha]',           'Red'),
    (65,  'Class 3 [101-1,000 ha]',        'Yellow'),
    (1,   'Class 4 [1,001-10,000 ha]',     'Orange'),
    (9,   'Class 5 [10,001-100,000 ha]',   'Brown'),
    (17,  'Class 6 [>100,000 ha]',         'Green'),
]

print(f"  {'Class':<30} {'Patches':>10} {'Pixels':>12} {'% FG':>8}")
print("  " + "-" * 65)
for val, label, color in class_labels:
    n_pch = patch_numb.get(val, 0)
    n_pxl = pxl_numb.get(val, 0)
    pct   = n_pxl / tot_fg * 100 if tot_fg > 0 else 0
    print(f"  {label:<30} {n_pch:>10} {n_pxl:>12} {pct:>8.2f}%")

In [ ]:
# --- Visualise Accounting output with GTB colormap ---
acc_tiff = acc_result.stats['output paths']['path tif']
data, cmap, norm = gtb_colormap(acc_tiff)

fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(data, cmap=cmap, norm=norm, interpolation='none')
ax.set_title('Accounting Result — Corsica\nCLC 2018 Forest/Non-Forest, 100m\nThresholds: 100, 1000, 10000, 100000, 1000000 pixels',
             fontsize=13, pad=15)
ax.axis('off')

# Legend
legend_patches = [
    mpatches.Patch(color='black',                                    label='Class 1: 1-10 ha'),
    mpatches.Patch(color='red',                                      label='Class 2: 11-100 ha'),
    mpatches.Patch(color='yellow',                                   label='Class 3: 101-1,000 ha'),
    mpatches.Patch(color='orange',                                   label='Class 4: 1,001-10,000 ha'),
    mpatches.Patch(color=(139/255, 90/255, 43/255),                  label='Class 5: 10,001-100,000 ha'),
    mpatches.Patch(color='green',                                    label='Class 6: >1,000,000 ha'),
    mpatches.Patch(color=(220/255, 220/255, 220/255),                label='Background'),
    mpatches.Patch(facecolor='white', edgecolor='black',             label='NoData'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

## 6. RSS — Raster Spatial Statistics

RSS computes patch-based connectivity indices that characterise the spatial structure and connectivity of the forest landscape:

| Index | Unit | Description |
|---|---|---|
| **CNOA** | pixels | Critical New Object Area — minimum patch size to increase connectivity |
| **ECA** | pixels | Equivalent Connected Area — area of a single maximally connected patch providing the same connectivity |
| **RAC** | % | Reference Area Coverage — forest proportion relative to total area |
| **COH** | % | Coherence — percentage of forest pixels effectively connected |
| **RPOT** | % | Restoration Potential — percentage of pixels that could improve connectivity |

In [ ]:
print("Running RSS...")
rss_result = pg.rss(
    in_tiff=fnf_tiff,
    outdir=OUT_DIR,
    stat_files=True,
    verb=False
)
print("\nRSS completed.")

In [ ]:
# --- RSS Statistics ---
print("Input pixel counts:")
for k, v in rss_result.stats['input stats'].items():
    print(f"  {k:<20}: {v:>10}")

print("\nPatch size statistics:")
out = rss_result.stats['output stats']
print(f"  Total patches      : {out['total patches']:>10}")
print(f"  Average patch size : {out['average patch size']:>10.1f} px")
print(f"  Median patch size  : {out['median patch size']:>10.1f} px")
print(f"  Largest patch      : {out['largest patch size']:>10} px")

print("\nConnectivity indices:")
print(f"  CNOA    : {out['CNOA']:>12.0f} px")
print(f"  ECA     : {out['ECA']:>12.0f} px")
print(f"  RAC     : {out['RAC']:>12.2f} %")
print(f"  COH     : {out['COH']:>12.2f} %")
print(f"  REST_POT: {out['REST_POT']:>12.2f} %")

## 7. Combined Overview

A side-by-side comparison of all four analysis outputs.

In [ ]:
# Load all output maps
mspa_tiff = mspa_result.stats['output paths']['path tif']
frag_tiff = frag_result.stats['output paths']['path tif']
acc_tiff  = acc_result.stats['output paths']['path tif']

mspa_data, mspa_cmap, mspa_norm = gtb_colormap(mspa_tiff)
frag_data, frag_cmap, frag_norm = gtb_colormap(frag_tiff)
acc_data,  acc_cmap,  acc_norm  = gtb_colormap(acc_tiff)

# Read input for reference
with rasterio.open(fnf_tiff) as src:
    fnf_data = src.read(1)

fnf_colors_list = ['white', 'lightgrey', 'darkgreen']
fnf_cmap_plt    = ListedColormap(fnf_colors_list)
fnf_norm_plt    = BoundaryNorm([0, 1, 2, 3], fnf_cmap_plt.N)

fig, axes = plt.subplots(1, 4, figsize=(24, 10))

# Input
axes[0].imshow(fnf_data, cmap=fnf_cmap_plt, norm=fnf_norm_plt, interpolation='none')
axes[0].set_title('Input\nForest/Non-Forest', fontsize=12)
axes[0].axis('off')

# MSPA
axes[1].imshow(mspa_data, cmap=mspa_cmap, norm=mspa_norm, interpolation='none')
axes[1].set_title('MSPA\nedge_width=1, conn=8', fontsize=12)
axes[1].axis('off')

# Fragmentation
axes[2].imshow(frag_data, cmap=frag_cmap, norm=frag_norm, interpolation='none')
axes[2].set_title('Fragmentation (FAD)\nwindow=27x27', fontsize=12)
axes[2].axis('off')

# Accounting
axes[3].imshow(acc_data, cmap=acc_cmap, norm=acc_norm, interpolation='none')
axes[3].set_title('Accounting\n5 size classes', fontsize=12)
axes[3].axis('off')

fig.suptitle('Forest Pattern Analysis — Corsica, CLC 2018 (100m)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Summary

In this notebook we have applied four complementary forest pattern analysis tools to the Corsica Forest/Non-Forest map:

- **MSPA** revealed the structural composition of the forest — identifying core areas, edges, perforations and connecting elements
- **Fragmentation (FAD)** quantified the local forest density across the landscape using a 2.7km moving window
- **Accounting** classified individual forest patches into 6 size classes from small isolated patches to large connected areas
- **RSS** provided landscape-scale connectivity indices including ECA, COH and RPOT

All output GeoTIFFs are saved in the output directory and can be opened directly in QGIS, ArcGIS or any GIS software.

In the next notebook we will apply the **Landscape Mosaic** analysis to the three-class land cover map.